# Ecommerce Data Exploration

This notebook is a reusable starting point for exploring an ecommerce dataset.

It covers:
- loading the most likely raw dataset from `data/raw/`
- inspecting schema, shape, and sample rows
- checking missing values and duplicates
- summarizing numeric and categorical columns
- visualizing distributions, correlations, and time trends when possible


## 1. Setup

If you have more than one file in `data/raw/`, set `DATA_FILE` explicitly in the next cell.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
SUPPORTED_SUFFIXES = {".csv", ".parquet"}
DATA_FILE = None


In [ ]:
candidate_files = []

if DATA_DIR.exists():
    candidate_files = sorted(
        path for path in DATA_DIR.iterdir() if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES
    )

if DATA_FILE is None:
    if len(candidate_files) == 1:
        data_path = candidate_files[0]
    elif len(candidate_files) > 1:
        raise ValueError(
            "Multiple raw data files found. Set DATA_FILE to one of: "
            + ", ".join(path.name for path in candidate_files)
        )
    else:
        raise FileNotFoundError(
            f"No CSV or Parquet files found in {DATA_DIR}. Add a dataset or set DATA_FILE manually."
        )
else:
    data_path = Path(DATA_FILE)
    if not data_path.is_absolute():
        data_path = PROJECT_ROOT / DATA_FILE

data_path


In [ ]:
if data_path.suffix.lower() == ".csv":
    df = pd.read_csv(data_path)
elif data_path.suffix.lower() == ".parquet":
    df = pd.read_parquet(data_path)
else:
    raise ValueError(f"Unsupported file format: {data_path.suffix}")

print(f"Loaded: {data_path.name}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
df.head()


## 2. Schema Overview

In [ ]:
schema_summary = pd.DataFrame(
    {
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "non_null_count": df.notna().sum().values,
        "null_count": df.isna().sum().values,
        "null_pct": (df.isna().mean() * 100).round(2).values,
        "unique_values": df.nunique(dropna=True).values,
    }
).sort_values(["null_pct", "unique_values"], ascending=[False, False])

schema_summary


In [ ]:
df.info()


## 3. Data Quality Checks

In [ ]:
row_count = len(df)
duplicate_rows = int(df.duplicated().sum())
duplicate_pct = round((duplicate_rows / row_count) * 100, 2) if row_count else 0

quality_summary = pd.DataFrame(
    {
        "metric": ["rows", "columns", "duplicate_rows", "duplicate_row_pct"],
        "value": [row_count, df.shape[1], duplicate_rows, duplicate_pct],
    }
)

quality_summary


In [ ]:
missing_values = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)
missing_values["missing_pct"] = ((missing_values["missing_count"] / len(df)) * 100).round(2)
missing_values = missing_values[missing_values["missing_count"] > 0]

missing_values if not missing_values.empty else "No missing values detected."


In [ ]:
if not missing_values.empty:
    plt.figure(figsize=(12, max(4, len(missing_values) * 0.35)))
    sns.barplot(
        data=missing_values.reset_index().rename(columns={"index": "column"}),
        x="missing_pct",
        y="column",
        palette="crest",
    )
    plt.title("Missing Values by Column")
    plt.xlabel("Missing Percentage")
    plt.ylabel("Column")
    plt.show()
else:
    print("No missing-value chart generated because the dataset is complete.")


## 4. Numeric Feature Exploration

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols


In [ ]:
if numeric_cols:
    df[numeric_cols].describe().T
else:
    print("No numeric columns found.")


In [ ]:
if numeric_cols:
    for column in numeric_cols[:6]:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        sns.histplot(df[column].dropna(), kde=True, ax=axes[0], color="#2a9d8f")
        axes[0].set_title(f"Distribution: {column}")
        sns.boxplot(x=df[column], ax=axes[1], color="#e9c46a")
        axes[1].set_title(f"Boxplot: {column}")
        plt.tight_layout()
        plt.show()
else:
    print("Skipping numeric charts because no numeric columns were found.")


In [ ]:
if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr(numeric_only=True)
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0)
    plt.title("Correlation Heatmap")
    plt.show()
else:
    print("Need at least two numeric columns to compute correlations.")


## 5. Categorical Feature Exploration

In [ ]:
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
categorical_cols


In [ ]:
if categorical_cols:
    categorical_summary = pd.DataFrame(
        {
            "column": categorical_cols,
            "unique_values": [df[col].nunique(dropna=True) for col in categorical_cols],
            "top_value": [df[col].mode(dropna=True).iat[0] if not df[col].mode(dropna=True).empty else np.nan for col in categorical_cols],
            "top_value_freq": [df[col].value_counts(dropna=True).iloc[0] if not df[col].value_counts(dropna=True).empty else 0 for col in categorical_cols],
        }
    ).sort_values("unique_values", ascending=False)
    categorical_summary
else:
    print("No categorical columns found.")


In [ ]:
if categorical_cols:
    for column in categorical_cols[:5]:
        top_counts = df[column].fillna("<missing>").value_counts().head(10)
        plt.figure(figsize=(12, 4))
        sns.barplot(x=top_counts.values, y=top_counts.index, palette="mako")
        plt.title(f"Top Categories: {column}")
        plt.xlabel("Count")
        plt.ylabel(column)
        plt.show()
else:
    print("Skipping categorical charts because no categorical columns were found.")


## 6. Datetime Checks and Trends

This step attempts to detect datetime-like columns and plot daily record volume for the first one that converts successfully.

In [ ]:
datetime_candidates = [
    col for col in df.columns
    if any(token in col.lower() for token in ["date", "time", "timestamp", "created", "updated", "order_at"])
]

converted_datetime_cols = []
for col in datetime_candidates:
    converted = pd.to_datetime(df[col], errors="coerce")
    if converted.notna().mean() >= 0.7:
        df[col] = converted
        converted_datetime_cols.append(col)

converted_datetime_cols


In [ ]:
if converted_datetime_cols:
    datetime_col = converted_datetime_cols[0]
    daily_volume = (
        df.dropna(subset=[datetime_col])
        .assign(day=df[datetime_col].dt.date)
        .groupby("day")
        .size()
        .rename("records")
    )

    daily_volume.plot(color="#264653")
    plt.title(f"Daily Record Volume Based on {datetime_col}")
    plt.xlabel("Date")
    plt.ylabel("Records")
    plt.show()
else:
    print("No reliable datetime columns were detected.")


## 7. Quick Takeaways

Use this section to capture observations after running the notebook.

- Which columns are the most important business metrics?
- Are there data quality issues that need cleaning before analysis?
- Which features look most promising for KPI tracking or modeling?
